In [2]:
import geemap
import ee
import geopandas as gpd
import os


In [3]:
ee.Initialize(project='ee-my-biswajit')

In [4]:
VB_PATH = r'D:\Personal Projects\Assignment\Village_Economic_Growth_Intelligence\data\output_data\vectore\filtered_village.shp'


In [5]:
village_gdf = gpd.read_file(VB_PATH)
village_gdf.head()

,id,objectid_1,objectid,village,vlcode,gram_panch,gram_pan_1,block,bkcode,subdistric,...,other_sour,nearest_to,nearest__1,shape_leng,shape_le_1,shape_area,ds_name,src_agency,state_name,geometry
0,3,3.0,450683.0,Jhawdanga Pt Iii,281737,JHAWDNGA,NaN,Mankachar,19,Mankachar,...,0.0,DHUBRI,65.0,5358.530345,5358.530345,1.262858e+06,Administrative Boundary,Survey of India (SOI),Assam\n,"POLYGON ((89.88414 25.5874, 89.88429 25.58214,..."
1,246,246.0,450926.0,River Block N C,281491,MUTHAKHOWA,NaN,Birshingjarua,14,South Salmara,...,0.0,DHUBRI,17.0,5167.716913,5167.716913,1.058195e+06,Administrative Boundary,Survey of India (SOI),Assam\n,"POLYGON ((89.90672 25.88832, 89.90569 25.88807..."
2,375,375.0,451055.0,East Atharokotharjhar Sareswar Beel,280441,NaN,NaN,Rupsi (Part),7,Bagribari (Pt),...,4.0,GAURIPUR,13.0,12441.998861,12441.998861,7.480253e+06,Administrative Boundary,Survey of India (SOI),Assam\n,"POLYGON ((89.95093 26.15301, 89.93556 26.13788..."
3,938,938.0,451618.0,Dholai Molai Pt Ix,301162,NaN,NaN,Katlicherra,171,Katlichara,...,0.0,LALA,15.0,4642.561591,4642.561591,1.106456e+06,Administrative Boundary,Survey of India (SOI),Assam\n,"POLYGON ((92.55724 24.43647, 92.55629 24.42878..."
4,18817,18817.0,471148.0,Doloi Jaroni,293343,AKHATGURI,NaN,Majuli,114,Majuli,...,0.0,JORHAT,0.0,5038.963940,5038.963940,1.363491e+06,Administrative Boundary,Survey of India (SOI),Assam\n,"POLYGON ((93.95919 26.84501, 93.95516 26.84491..."


In [6]:
ee_village_fc = geemap.gdf_to_ee(village_gdf)

In [7]:
# Annual VIIRS collections
viirs_v21 = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V21")
viirs_v22 = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V22")

# Select 2020 from V2.1 and 2025 from V2.2
image_2020 = ee.Image(
    viirs_v21.filter(
        ee.Filter.eq("system:index", "20200101")
    ).first()
)

image_2025 = ee.Image(
    viirs_v22.filter(
        ee.Filter.eq("system:index", "20250101")
    ).first()
)

In [8]:
image_2025

In [9]:
bands = [
    "average",
    "average_masked",
    "median",
    "median_masked",
    "maximum",
    "cf_cvg",
    "cvg"
]

# Rename bands to distinguish years
image_2020 = image_2020.select(
    bands,
    [f"{band}_2020" for band in bands]
)

image_2025 = image_2025.select(
    bands,
    [f"{band}_2025" for band in bands]
)

# Combine both years into one multiband image
viirs_stack = image_2020.addBands(image_2025)

print("Final bands:", viirs_stack.bandNames().getInfo())

Final bands: ['average_2020', 'average_masked_2020', 'median_2020', 'median_masked_2020', 'maximum_2020', 'cf_cvg_2020', 'cvg_2020', 'average_2025', 'average_masked_2025', 'median_2025', 'median_masked_2025', 'maximum_2025', 'cf_cvg_2025', 'cvg_2025']


In [ ]:
viirs_stats = viirs_stack.reduceRegions(
    collection=ee_village_fc,
    reducer=ee.Reducer.mean(),
    scale=463.83,
    tileScale=4
)

# print("Number of villages:", viirs_stats.size().getInfo())

EEException: Request payload size exceeds the limit: 10485760 bytes.

In [11]:
viirs_gdf = geemap.ee_to_gdf(viirs_stats)

viirs_gdf.to_file(r"D:\Personal Projects\Assignment\Village_Economic_Growth_Intelligence\data\output_data\vectore\village_viirs_2020_2024.gpkg", driver="GPKG")


EEException: Request payload size exceeds the limit: 10485760 bytes.

In [13]:
selectors = (
    ["village_id"] +
    [f"{band}_{year}" for year in [2020, 2025] for band in bands]
)

task = ee.batch.Export.table.toDrive(
    collection=viirs_stats,
    description="VIIRS_Village_2020_2025",
    folder="GEE_Exports",
    fileNamePrefix="village_viirs_2020_2025",
    fileFormat="CSV",
    selectors=selectors
)

task.start()

print(task.status())

EEException: Request payload size exceeds the limit: 10485760 bytes.